In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json
import re
import ast
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
BASE = Path("drive/MyDrive/Yelp JSON/yelp_dataset")

In [4]:
PROCESSED   = Path(BASE/"data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# Business

In [5]:
RAW_BUSINESS = BASE / "yelp_academic_dataset_business.json"

In [6]:
df_raw = pd.read_json(RAW_BUSINESS, lines=True)

In [7]:
print(f"Shape raw: {df_raw.shape}")
print(f"Columns:  {df_raw.columns.tolist()}")

Shape raw: (150346, 14)
Columns:  ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']


In [8]:
def parse_inner_dict(val):
    """Convert strings "{'garage': False}" as a real dict."""
    if isinstance(val, dict):
        return val
    if isinstance(val, str):
        # Yelp uses simple quotation marks in serialized strings → ast.literal_eval
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, dict):
                return parsed
        except (ValueError, SyntaxError):
            pass
        # Some values are serialized booleans: "True", "False", "None"
        if val == "True":
            return True
        if val == "False":
            return False
        if val == "None":
            return None
        # Categories as "u'casual'"  →  clean
        cleaned = re.sub(r"^u'|'$", "", val).strip("'\"")
        return cleaned if cleaned else None
    return val

In [ ]:
def flatten_attributes(attr_dict):
    """
    Gets business attributes dict and returns a plain dict

    strategy:
    - Simple keys     →  attr__KeyName
    - Keys with subdict →  attr__KeyName__SubKey
    """
    if not isinstance(attr_dict, dict):
        return {}

    flat = {}
    for key, val in attr_dict.items():
        parsed = parse_inner_dict(val)

        if isinstance(parsed, dict):
            # Sub-level: ex. BusinessParking → {garage: F, lot: T, ...}
            for subkey, subval in parsed.items():
                col = f"attr__{key}__{subkey}"
                flat[col] = parse_inner_dict(subval)
        else:
            flat[f"attr__{key}"] = parsed

    return flat

In [10]:
attrs_expanded = df_raw["attributes"].apply(
    lambda x: flatten_attributes(x) if isinstance(x, dict) else {}
)
df_attrs = pd.DataFrame(attrs_expanded.tolist(), index=df_raw.index)

In [11]:
print(f"Attributes generated columns: {df_attrs.shape[1]}")
print(f"Average Missingness (attrs): {df_attrs.isna().mean().mean():.1%}")

Attributes generated columns: 88
Average Missingness (attrs): 84.4%


In [12]:
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def parse_hours(hours_dict):
    """
    Converts {"Monday": "9:0-21:0"} to numeric columns.

    Returns how many days have registered hours (0–7)
    and if it opens on the weekends.
    """
    result = {
        "hours__open_days": 0,
        "hours__has_weekend": 0,
        **{f"hours__open_{day.lower()}": 0 for day in DAYS}
    }

    if not isinstance(hours_dict, dict):
        return result

    result["hours__open_days"] = len(hours_dict)
    result["hours__has_weekend"] = int("Saturday" in hours_dict or "Sunday" in hours_dict)

    # Boolean column per day (useful if we do temporal clustering)
    for day in DAYS:
        result[f"hours__open_{day.lower()}"] = int(day in hours_dict)

    return result

In [13]:
hours_expanded = df_raw["hours"].apply(parse_hours)
df_hours = pd.DataFrame(hours_expanded.tolist(), index=df_raw.index)
print(f"Generated columns per hours: {df_hours.shape[1]}")

Generated columns per hours: 9


In [14]:
def parse_categories(cat_str):
    """Converts "Restaurants, Italian, Pizza" → ["Restaurants", "Italian", "Pizza"]"""
    if not isinstance(cat_str, str) or not cat_str.strip():
        return []
    return [c.strip() for c in cat_str.split(",") if c.strip()]

In [15]:
df_raw["categories_list"] = df_raw["categories"].apply(parse_categories)
df_raw["primary_category"] = df_raw["categories_list"].apply(
    lambda lst: lst[0] if lst else None
)
df_raw["n_categories"] = df_raw["categories_list"].apply(len)

In [16]:
df_raw.head(2)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours,categories_list,primary_category,n_categories
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None,"[Doctors, Traditional Chinese Medicine, Naturo...",Doctors,6
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ...","[Shipping Centers, Local Services, Notaries, M...",Shipping Centers,5


In [17]:
a = df_raw["primary_category"].value_counts()

In [18]:
a.shape

(1159,)

In [ ]:
# we create a df with business unique id + each of their categories
bridge = (
    df_raw[["business_id", "categories_list"]]
    .explode("categories_list")
    .rename(columns={"categories_list": "category"})
    .dropna(subset=["category"])
    .reset_index(drop=True)
)
print(f"Bridge rows: {len(bridge):,}  |  unique categories: {bridge['category'].nunique()}")

Bridge rows: 668,592  |  unique categories: 1311


In [25]:
df_attrs.head()

,attr__ByAppointmentOnly,attr__BusinessAcceptsCreditCards,attr__BikeParking,attr__RestaurantsPriceRange2,attr__CoatCheck,attr__RestaurantsTakeOut,attr__RestaurantsDelivery,attr__Caters,attr__WiFi,attr__BusinessParking__garage,...,attr__HairSpecializesIn,attr__Music,attr__DietaryRestrictions__dairy-free,attr__DietaryRestrictions__gluten-free,attr__DietaryRestrictions__vegan,attr__DietaryRestrictions__kosher,attr__DietaryRestrictions__halal,attr__DietaryRestrictions__soy-free,attr__DietaryRestrictions__vegetarian,attr__DietaryRestrictions
0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,True,True,2,False,False,False,False,no,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,False,True,1,NaN,True,False,True,free,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,True,True,NaN,NaN,True,NaN,False,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df_raw.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours', 'categories_list',
       'primary_category', 'n_categories'],
      dtype='object')

In [19]:
base_cols = [
    "business_id", "name", "city", "state",
    "latitude", "longitude", "stars", "review_count", "is_open"
]

In [20]:
business_clean = pd.concat(
    [
        df_raw[base_cols].copy(),
        df_raw[["categories_list", "primary_category", "n_categories"]].copy(),
        df_attrs.copy(),
        df_hours.copy()
    ],
    axis=1
)

In [21]:
attr_cols = [c for c in business_clean.columns if c.startswith("attr__")]
drop_cols = [c for c in attr_cols if business_clean[c].isna().mean() > 0.85]
drop_cols

['attr__CoatCheck',
 'attr__HappyHour',
 'attr__DogsAllowed',
 'attr__BusinessParking',
 'attr__Ambience',
 'attr__RestaurantsTableService',
 'attr__DriveThru',
 'attr__BusinessAcceptsBitcoin',
 'attr__Smoking',
 'attr__Music__dj',
 'attr__Music__background_music',
 'attr__Music__no_music',
 'attr__Music__jukebox',
 'attr__Music__live',
 'attr__Music__video',
 'attr__Music__karaoke',
 'attr__GoodForDancing',
 'attr__AcceptsInsurance',
 'attr__BestNights__monday',
 'attr__BestNights__tuesday',
 'attr__BestNights__friday',
 'attr__BestNights__wednesday',
 'attr__BestNights__thursday',
 'attr__BestNights__sunday',
 'attr__BestNights__saturday',
 'attr__BYOB',
 'attr__Corkage',
 'attr__BYOBCorkage',
 'attr__HairSpecializesIn__straightperms',
 'attr__HairSpecializesIn__coloring',
 'attr__HairSpecializesIn__extensions',
 'attr__HairSpecializesIn__africanamerican',
 'attr__HairSpecializesIn__curly',
 'attr__HairSpecializesIn__kids',
 'attr__HairSpecializesIn__perms',
 'attr__HairSpecializesIn

In [26]:
len(attr_cols)

88

In [25]:
len(drop_cols)

51

In [ ]:
# dropping colummns with more than 85% of missing data = drop_cols
business_clean = business_clean.drop(columns=drop_cols)

In [ ]:
bool_like_cols = [c for c in business_clean.columns if c.startswith("attr__")]
# convert strings true/false to actual booleans
for c in bool_like_cols:
    vals = set(business_clean[c].dropna().astype(str).str.lower().unique())
    if vals.issubset({"true", "false"}):
        business_clean[c] = (
            business_clean[c]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
            .astype("boolean")
        )

In [ ]:
# print info of business df to view nulls and types
business_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 58 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   business_id                       150346 non-null  object 
 1   name                              150346 non-null  object 
 2   city                              150346 non-null  object 
 3   state                             150346 non-null  object 
 4   latitude                          150346 non-null  float64
 5   longitude                         150346 non-null  float64
 6   stars                             150346 non-null  float64
 7   review_count                      150346 non-null  int64  
 8   is_open                           150346 non-null  int64  
 9   categories_list                   150346 non-null  object 
 10  primary_category                  150243 non-null  object 
 11  n_categories                      150346 non-null  i

In [ ]:
# Converting objet to Category
for c in business_clean.select_dtypes(include="object").columns:
    if c.startswith("attr__"):
        nunq = business_clean[c].nunique(dropna=True)
        if nunq <= 20:
            business_clean[c] = business_clean[c].astype("category")

In [ ]:
# null values are left on attr columns because they are not necessary False or 0 and may be just unreported data
# we dont fill these values to prevent wrong data for businesses
business_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 58 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   business_id                       150346 non-null  object  
 1   name                              150346 non-null  object  
 2   city                              150346 non-null  object  
 3   state                             150346 non-null  object  
 4   latitude                          150346 non-null  float64 
 5   longitude                         150346 non-null  float64 
 6   stars                             150346 non-null  float64 
 7   review_count                      150346 non-null  int64   
 8   is_open                           150346 non-null  int64   
 9   categories_list                   150346 non-null  object  
 10  primary_category                  150243 non-null  object  
 11  n_categories                      15034

In [ ]:
# saving both business and bridge
business_clean.to_parquet(PROCESSED / "business_clean.parquet", index=False)
bridge.to_parquet(PROCESSED / "business_categories.parquet", index=False)

In [6]:
business_clean = pd.read_parquet(PROCESSED / "business_clean.parquet")

# Reviews

In [7]:
RAW_REVIEWS = BASE / "yelp_academic_dataset_review.json"

In [ ]:
chunks = []
business_ids = set(business_clean["business_id"])
# reading them in chunks because the original .json provided is large
for chunk in pd.read_json(RAW_REVIEWS, lines=True, chunksize=100000):
    filtered = chunk[chunk["business_id"].isin(business_ids)]
    chunks.append(filtered)

review_clean = pd.concat(chunks, ignore_index=True)

In [ ]:
# visualize columns and types of data it has
review_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6990280 entries, 0 to 6990279
Data columns (total 9 columns):
 #   Column       Dtype         
---  ------       -----         
 0   review_id    object        
 1   user_id      object        
 2   business_id  object        
 3   stars        int64         
 4   useful       int64         
 5   funny        int64         
 6   cool         int64         
 7   text         object        
 8   date         datetime64[ns]
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 480.0+ MB


In [15]:
review_clean.isna().mean()

,0
review_id,0.0
user_id,0.0
business_id,0.0
stars,0.0
useful,0.0
funny,0.0
cool,0.0
text,0.0
date,0.0


In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

output_path = PROCESSED / "review_clean.parquet"
business_ids = set(business_clean["business_id"])

writer = None

cols = [
    "review_id", "user_id", "business_id", "stars",
    "text", "date", "useful", "funny", "cool"
]

for chunk in pd.read_json(RAW_REVIEWS, lines=True, chunksize=100000):
    filtered = chunk[chunk["business_id"].isin(business_ids)][cols].copy()
    # we filter to make sure its only saving reviews linked to the df of business just cleaned
    if filtered.empty:
        continue

    filtered["date"] = pd.to_datetime(filtered["date"], errors="coerce")
    filtered["text_len"] = filtered["text"].str.len()

    table = pa.Table.from_pandas(filtered, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema, compression="snappy")

    writer.write_table(table)

if writer is not None:
    writer.close()

In [ ]:
# we filter the df with only the relevant columns
test = pd.read_parquet(
    PROCESSED / "review_clean.parquet",
    columns=["user_id", "business_id", "stars"]
)
print(test.shape)
print(test.head())

(6990280, 3)
                  user_id             business_id  stars
0  mh_-eMZ6K5RLWhZyISBhwA  XQfwVwDr-v0ZS3_CbbE5Xw      3
1  OyoGAe7OKpv6SyGZT5g77Q  7ATYjTIgM3jUlt4UM3IypQ      5
2  8g_iMtfSiwikVnbP2etR0A  YjUWPpI6HXG530lwP-fb2A      3
3  _7bHUi9Uuf5__HHc_Q8guQ  kxX2SOes4o-D3ZQBkiMRfA      5
4  bcjbaE6dDog4jkNY91ncLQ  e4Vwtrqf-wpJfwesgvdgxQ      4


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6990280 entries, 0 to 6990279
Data columns (total 3 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   user_id      object
 1   business_id  object
 2   stars        int64 
dtypes: int64(1), object(2)
memory usage: 160.0+ MB


In [11]:
test.isna().mean()

,0
user_id,0.0
business_id,0.0
stars,0.0


In [12]:
# saves test(interaction - stars w business) with snappy compression
test.to_parquet(PROCESSED / "review_interactions.parquet", compression="snappy")

# Users

In [7]:
RAW_USERS = BASE / "yelp_academic_dataset_user.json"

In [7]:
review_interactions = pd.read_parquet(PROCESSED / "review_interactions.parquet")

In [ ]:
# read original .json to visualize what columns it has and which we are going to be saving
df_user = pd.read_json(RAW_USERS, lines=True, nrows=1)
print(df_user.T)

                                                                    0
user_id                                        qVc8ODYU5SZjKXVBgXdI7w
name                                                           Walker
review_count                                                      585
yelping_since                                     2007-01-25 16:47:26
useful                                                           7217
funny                                                            1259
cool                                                             5994
elite                                                            2007
friends             NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...
fans                                                              267
average_stars                                                    3.91
compliment_hot                                                    250
compliment_more                                                    65
compliment_profile  

In [ ]:
out_path = PROCESSED / "user_clean.parquet"
writer = None
buffer = []

bad_lines = 0

with open(RAW_USERS, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue

        try:
            row = json.loads(line, strict=False)
        except json.JSONDecodeError:
            bad_lines += 1
            continue
        # we read and save only userid, hoy many reviews the user has done, average stars the person gives and date when they joined YELP
        buffer.append({
            "user_id": row.get("user_id"),
            "review_count": row.get("review_count"),
            "average_stars": row.get("average_stars"),
            "yelping_since": row.get("yelping_since"),
        })

        if len(buffer) >= 10000:
            df = pd.DataFrame(buffer)
            df["yelping_since"] = pd.to_datetime(df["yelping_since"], errors="coerce")

            table = pa.Table.from_pandas(df, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")

            writer.write_table(table)

            buffer = []
            del df, table
            gc.collect()

if buffer:
    df = pd.DataFrame(buffer)
    df["yelping_since"] = pd.to_datetime(df["yelping_since"], errors="coerce")
    table = pa.Table.from_pandas(df, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")

    writer.write_table(table)

    del df, table
    gc.collect()

if writer:
    writer.close()

print("Bad lines skipped:", bad_lines)

Bad lines skipped: 1


In [11]:
df_user = pd.read_parquet(PROCESSED / "user_clean.parquet")

In [12]:
df_user.head(2)

,user_id,review_count,average_stars,yelping_since
0,qVc8ODYU5SZjKXVBgXdI7w,585,3.91,2007-01-25 16:47:26
1,j14WgRoU_-2ZE1aw1dXrJg,4333,3.74,2009-01-25 04:35:42


# Checkin

In [15]:
df_checkin = pd.read_json(BASE/"yelp_academic_dataset_checkin.json", lines=True)

In [16]:
df_checkin.head(2)

,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."


In [17]:
df_checkin.shape

(131930, 2)

In [18]:
business_clean = pd.read_parquet(PROCESSED / "business_clean.parquet")

In [19]:
business_ids = set(business_clean["business_id"])

In [20]:
checkin_clean = df_checkin[df_checkin["business_id"].isin(business_ids)].copy()

In [21]:
checkin_clean.shape

(131930, 2)

In [ ]:
# we read and save only how many checkins has a business made 
checkin_clean["n_checkins"] = checkin_clean["date"].apply(
    lambda x: len(x.split(",")) if isinstance(x, str) else 0
)

checkin_clean = checkin_clean[["business_id", "n_checkins"]]

In [23]:
checkin_clean.head(2)

,business_id,n_checkins
0,---kPU91CF4Lq2-WlRu9Lw,11
1,--0iUa4sNDFiZFrAdIWhZQ,10


In [24]:
checkin_clean.to_parquet(PROCESSED / "checkin_clean.parquet", index=False)

# Scale Analysis

In [ ]:
PROCESSED = Path("drive/MyDrive/Yelp JSON/yelp_dataset/data/processed")

files = {
    "business_clean":       PROCESSED / "business_clean.parquet",
    "business_categories":  PROCESSED / "business_categories.parquet",
    "review_interactions":  PROCESSED / "review_interactions.parquet",
    "user_clean":           PROCESSED / "user_clean.parquet",
    "checkin_clean":        PROCESSED / "checkin_clean.parquet",
}

for name, path in files.items():
    if not path.exists():
        print(f"\n{name}: archivo no encontrado en {path}")
        continue

    df = pd.read_parquet(path)
    rows, cols = df.shape
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    file_mb = path.stat().st_size / 1e6
    miss = df.isna().mean()
    cols_with_null = (miss > 0).sum()

    print(f"\n {name} ")
    print(f"  Filas:              {rows:>12,}")
    print(f"  Columnas:           {cols:>12,}")
    print(f"  Memoria RAM:        {mem_mb:>11.1f} MB")
    print(f"  Tamaño en disco:    {file_mb:>11.1f} MB")
    print(f"  Columnas con nulos: {cols_with_null:>12,}  de {cols}")

    if cols_with_null > 0:
        top_miss = miss[miss > 0].sort_values(ascending=False).head(5)
        print("  Top columnas con más nulos:")
        for col, pct in top_miss.items():
            print(f"    {col:<45} {pct:.1%}")

    del df

print("\n Sparsity of user-item matrix (review_interactions) ")
ri = pd.read_parquet(PROCESSED / "review_interactions.parquet")
n_users     = ri["user_id"].nunique()
n_businesses = ri["business_id"].nunique()
n_interactions = len(ri)
sparsity = 1 - (n_interactions / (n_users * n_businesses))
print(f"  Usuarios únicos:     {n_users:>12,}")
print(f"  Negocios únicos:     {n_businesses:>12,}")
print(f"  Interacciones:       {n_interactions:>12,}")
print(f"  Densidad matrix:     {(1-sparsity)*100:>11.4f}%")
print(f"  Sparsity:            {sparsity*100:>11.4f}%")
del ri


 business_clean 
  Filas:                   150,346
  Columnas:                     58
  Memoria RAM:               92.9 MB
  Tamaño en disco:           12.1 MB
  Columnas con nulos:           38  de 58
  Top columnas con más nulos:
    attr__GoodForMeal__dessert                    83.9%
    attr__GoodForMeal__latenight                  83.3%
    attr__GoodForMeal__brunch                     83.2%
    attr__GoodForMeal__breakfast                  82.5%
    attr__GoodForMeal__lunch                      82.4%

 business_categories 
  Filas:                   668,592
  Columnas:                      2
  Memoria RAM:               88.1 MB
  Tamaño en disco:            5.3 MB
  Columnas con nulos:            0  de 2

 review_interactions 
  Filas:                 6,990,280
  Columnas:                      3
  Memoria RAM:             1048.5 MB
  Tamaño en disco:          184.1 MB
  Columnas con nulos:            0  de 3

 user_clean 
  Filas:                   116,430
  Columnas:          